# Finetuning with Azure OpenAI

This notebook demonstrates how to fine-tune language models using **Supervised Fine-Tuning (SFT)**, **Direct Preference Optimization (DPO)**, and **Reinforcement Fine-Tuning (RFT)**.

**Note**: Execute each cell in sequence.

## 1. Setup and Installation

In [ ]:
%pip install -q \
  "azure-ai-projects>=2.0.0b1" \
  openai \
  azure-identity \
  azure-mgmt-cognitiveservices \
  "azure-ai-evaluation>=1.13.0" \
  python-dotenv

## 2. Configure Azure

This sets up the Azure credential authentication and initializes OpenAI client needed for fine-tuning workflows.

In [ ]:
import os
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.cognitiveservices.models import Deployment, DeploymentProperties, DeploymentModel, Sku

In [ ]:
# TODO: Update RESOURCE_GROUP name and OPENAI_API_KEY that have been provided to your team via gradescope.
# Do not modify other fields.
RESOURCE_GROUP = 'cis-5270-team-9'
OPENAI_API_KEY = ''
OPENAI_ENDPOINT = f"https://{RESOURCE_GROUP}.openai.azure.com"
SUBSCRIPTION_ID = ''

In [ ]:
# Azure resource targeting
os.environ["AZURE_SUBSCRIPTION_ID"] = SUBSCRIPTION_ID
os.environ["AZURE_RESOURCE_GROUP"] = RESOURCE_GROUP
os.environ["AZURE_AOAI_ACCOUNT"] = RESOURCE_GROUP
os.environ["AZURE_OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = OPENAI_ENDPOINT
CREDENTIAL = DefaultAzureCredential()

We're using **gpt-4.1-nano** in this example, but you can use other supported GPT models.

In [ ]:
openai_client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_ENDPOINT,
    api_version="2025-04-01-preview",
)
model_name = 'gpt-4.1-nano-2025-04-14'
print("Connected to Azure OpenAI")

## 3. Upload Training Files

Upload the training and validation JSONL files to Microsoft Foundry. Each file is assigned a unique ID that will be referenced when creating the fine-tuning job. See [here]() for how dataset should be prepared.

In [ ]:
from datasets import load_dataset
import json

# ----------------------------
# Config
# ----------------------------
DATASET_NAME = "saurabh5/rlvr-code-data-Haskell"

TRAIN_OUTPUT_PATH = "training_raw.jsonl"
VALIDATION_OUTPUT_PATH = "validation_raw.jsonl"
TEST_OUTPUT_PATH = "test_raw.jsonl"

NUM_EXAMPLES = 2000
VAL_FRAC = 0.1
TEST_FRAC = 0.1
SEED = 42

# ----------------------------
# Load dataset
# ----------------------------
ds = load_dataset(DATASET_NAME)
base_ds = ds["train"]

# Use the same shuffle + subset logic as before
subset = base_ds.shuffle(seed=SEED).select(range(NUM_EXAMPLES))

# First split off test
split1 = subset.train_test_split(test_size=TEST_FRAC, seed=SEED)
train_val_ds = split1["train"]
test_ds = split1["test"]

# Then split train/val from remaining data
val_relative_frac = VAL_FRAC / (1 - TEST_FRAC)
split2 = train_val_ds.train_test_split(test_size=val_relative_frac, seed=SEED)
train_ds = split2["train"]
val_ds = split2["test"]

print(f"Train: {len(train_ds)}")
print(f"Val:   {len(val_ds)}")
print(f"Test:  {len(test_ds)}")

# ----------------------------
# Write raw JSONL
# ----------------------------
def write_raw_jsonl(dataset, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for ex in dataset:
            # Keep the example exactly as-is, including test cases
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

write_raw_jsonl(train_ds, TRAIN_OUTPUT_PATH)
write_raw_jsonl(val_ds, VALIDATION_OUTPUT_PATH)
write_raw_jsonl(test_ds, TEST_OUTPUT_PATH)

print("Wrote raw train/val/test files.")

In [ ]:
from datasets import load_dataset
import json

# ----------------------------
# Config
# ----------------------------
DATASET_NAME = "saurabh5/rlvr-code-data-Haskell"

TRAIN_OUTPUT_PATH = "training.jsonl"
VALIDATION_OUTPUT_PATH = "validation.jsonl"
TEST_OUTPUT_PATH = "test.jsonl"

NUM_EXAMPLES = 2000
VAL_FRAC = 0.1
TEST_FRAC = 0.1
SEED = 42

# ----------------------------
# Load dataset
# ----------------------------
ds = load_dataset(DATASET_NAME)
base_ds = ds["train"]

subset = base_ds.shuffle(seed=SEED).select(range(NUM_EXAMPLES))

# First split off test
split1 = subset.train_test_split(test_size=TEST_FRAC, seed=SEED)
train_val_ds = split1["train"]
test_ds = split1["test"]

# Then split train/val from remaining data
val_relative_frac = VAL_FRAC / (1 - TEST_FRAC)
split2 = train_val_ds.train_test_split(test_size=val_relative_frac, seed=SEED)
train_ds = split2["train"]
val_ds = split2["test"]

print(f"Train: {len(train_ds)}")
print(f"Val:   {len(val_ds)}")
print(f"Test:  {len(test_ds)}")

def write_foundry_jsonl(dataset, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for ex in dataset:
            example = {
                "messages": [
                    {"role": "user", "content": ex["translated_problem"]},
                    {"role": "assistant", "content": ex["translated_solution"]},
                ]
            }
            f.write(json.dumps(example, ensure_ascii=False) + "\n")

def write_test_jsonl(dataset, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for ex in dataset:
            example = {
                "prompt": ex["translated_problem"],
                "reference_solution": ex["translated_solution"],
            }

            # keep any extra fields if they exist
            for k in ex.keys():
                if k not in ["translated_problem", "translated_solution"]:
                    example[k] = ex[k]

            f.write(json.dumps(example, ensure_ascii=False) + "\n")

write_foundry_jsonl(train_ds, TRAIN_OUTPUT_PATH)
write_foundry_jsonl(val_ds, VALIDATION_OUTPUT_PATH)
write_test_jsonl(test_ds, TEST_OUTPUT_PATH)

print("Wrote train/val/test files.")

In [ ]:
# Define dataset file paths
training_file_path = "training.jsonl"
validation_file_path = "validation.jsonl"

In [ ]:
print("Uploading training file...")
with open(training_file_path, "rb") as f:
    train_file = openai_client.files.create(file=f, purpose="fine-tune")

print("Uploading validation file...")
with open(validation_file_path, "rb") as f:
    validation_file = openai_client.files.create(file=f, purpose="fine-tune")

train_file_id = train_file.id
val_file_id = validation_file.id

print(f"Training file ID: {train_file_id}")
print(f"Validation file ID: {val_file_id}")

Microsoft Foundry needs to process the uploaded files before they can be used for fine-tuning.

In [ ]:
print("Waiting for files to be processed...")
openai_client.files.wait_for_processing(train_file_id)
openai_client.files.wait_for_processing(val_file_id)
print("Files ready!")

## 4. Create a Fine-Tuning Job

Create a fine-tuning job with your uploaded datasets. Configure the following hyperparameters to control the training process:

**Hyperparameters:**
1. **n_epochs (1)**: Number of complete passes through the training dataset. More epochs can improve performance but may lead to overfitting. Typical range: 1-10.
2. **batch_size (1)**: Number of training examples processed together in each iteration. Smaller batches provide more frequent updates. Typical range: 1-8.
3. **learning_rate_multiplier (1.0)**: Scales the default learning rate. Values < 1.0 make training more conservative, while values > 1.0 speed up learning but may cause instability. Typical range: 0.1-2.0.

**Note**: Adjust these based on your dataset size and quality.

### 4-1. Supervised Fine-Tuning

In [ ]:
model_name = 'gpt-4.1-mini-2025-04-14'
print(f"Creating supervised fine-tuning job for {model_name}")

fine_tune_job = openai_client.fine_tuning.jobs.create(
    model=model_name,
    training_file=train_file_id,
    validation_file=val_file_id,
    method={
        "type": "supervised",
        "supervised": {"hyperparameters": {"n_epochs": 1, "batch_size": 8, "learning_rate_multiplier": 1.0}},
    },
    extra_body={"trainingType": "GlobalStandard"},
    suffix="supervised-fine-tuning"
)

print(f"Fine-tuning job created!")
print(f"Job ID: {fine_tune_job.id}")
print(f"Status: {fine_tune_job.status}")
print(f"Model: {fine_tune_job.model}")

In [ ]:
jobs = openai_client.fine_tuning.jobs.list(limit=50)

for job in jobs.data:
    print(job)

In [ ]:
job = openai_client.fine_tuning.jobs.retrieve("ftjob-dbcecaa57a274ce29bbfab46a741ded0")

print(job.fine_tuned_model)

In [ ]:
import os
import json
import time
from openai import AzureOpenAI
from tqdm.auto import tqdm

TEST_INPUT_PATH = "test.jsonl"

# Map deployments to desired output suffixes
DEPLOYMENTS = {
    "1-mini-2025-04-14-haskell-dpo-c9e34": "dpo-nano",
}

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version="2025-03-01-preview",
)

SYSTEM_PROMPT = """You are a Haskell coding assistant.
Return only Haskell code.
Do not include markdown fences.
Do not include explanations."""

def generate_completion(prompt: str, deployment: str) -> str:
    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content.strip()

with open(TEST_INPUT_PATH, "r", encoding="utf-8") as f:
    examples = [json.loads(line) for line in f]

for deployment, suffix in DEPLOYMENTS.items():
    output_path = f"test_predictions_{suffix}.jsonl"
    num_errors = 0

    with open(output_path, "w", encoding="utf-8") as f_out:
        for ex in tqdm(examples, desc=f"Generating predictions: {suffix}"):
            try:
                pred = generate_completion(ex["prompt"], deployment)
            except Exception as e:
                pred = ""
                num_errors += 1
                print(f"\nError on {suffix}: {e}")
                time.sleep(2)

            out = {
                "deployment": deployment,
                "prompt": ex["prompt"],
                "reference_solution": ex["reference_solution"],
                "prediction": pred,
            }

            for k, v in ex.items():
                if k not in out:
                    out[k] = v

            f_out.write(json.dumps(out, ensure_ascii=False) + "\n")

    print(f"Saved predictions to {output_path}")
    print(f"Errors for {suffix}: {num_errors}")

**Evaluation**

In [ ]:
!apt-get update
!apt-get install -y ghc

In [ ]:
import json
import subprocess
import tempfile
import os
from tqdm.auto import tqdm

TEST_RAW_PATH = "test_raw.jsonl"
PRED_PATH = "test_predictions_dpo-nano.jsonl"
RESULTS_PATH = "dpo-nano.json"

MAX_EXAMPLES = 200
TIMEOUT_SECONDS = 20
PRINT_EVERY = 20

COMMON_SOLUTION_IMPORTS = """module Solution where

import Data.List
import Data.Ord
import Data.Maybe
import Data.Char
import Data.Function
import qualified Data.Map.Strict as Map
import qualified Data.Map as MapLazy
import qualified Data.Set as Set
import qualified Data.List as List
import qualified Data.Ord as Ord
import System.IO.Unsafe
import Data.IORef
"""

def run_tests(solution_code, tests):
    with tempfile.TemporaryDirectory() as tmpdir:
        solution_path = os.path.join(tmpdir, "Solution.hs")
        main_path = os.path.join(tmpdir, "Main.hs")

        solution_module = COMMON_SOLUTION_IMPORTS + "\n" + solution_code

        with open(solution_path, "w", encoding="utf-8") as f:
            f.write(solution_module)

        test_defs = []
        test_checks = []

        for i, test in enumerate(tests):
            test = test.strip()
            test_defs.append(f"test_{i} :: Bool")
            test_defs.append(f"test_{i} = ({test})")
            test_defs.append("")
            test_checks.append(
                f'  putStrLn $ (if test_{i} then "PASS_{i}" else "FAIL_{i}")'
            )

        if not test_checks:
            test_checks = ['  putStrLn "NO_TESTS"']

        main_code = "\n".join([
            "module Main where",
            "import Solution",
            "import qualified Data.Map.Strict as Map",
            "import qualified Data.Map as MapLazy",
            "import qualified Data.Set as Set",
            "import qualified Data.List as List",
            "import qualified Data.Ord as Ord",
            "",
            *test_defs,
            "main :: IO ()",
            "main = do",
            *test_checks
        ])

        with open(main_path, "w", encoding="utf-8") as f:
            f.write(main_code)

        try:
            result = subprocess.run(
                ["runghc", "-i" + tmpdir, main_path],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=TIMEOUT_SECONDS,
            )
        except subprocess.TimeoutExpired:
            return {
                "compiled": False,
                "passed": 0,
                "total": len(tests),
                "stdout": "",
                "stderr": f"Execution timed out after {TIMEOUT_SECONDS} seconds.",
                "main_code": main_code,
            }

        if result.returncode != 0:
            return {
                "compiled": False,
                "passed": 0,
                "total": len(tests),
                "stdout": result.stdout,
                "stderr": result.stderr,
                "main_code": main_code,
            }

        passed = sum(
            1 for line in result.stdout.splitlines()
            if line.startswith("PASS_")
        )

        return {
            "compiled": True,
            "passed": passed,
            "total": len(tests),
            "stdout": result.stdout,
            "stderr": result.stderr,
            "main_code": main_code,
        }

with open(TEST_RAW_PATH, "r", encoding="utf-8") as f:
    test_examples = [json.loads(line) for line in f]

with open(PRED_PATH, "r", encoding="utf-8") as f:
    pred_examples = [json.loads(line) for line in f]

if len(test_examples) != len(pred_examples):
    print(
        f"Warning: test file has {len(test_examples)} examples but prediction file has {len(pred_examples)} examples."
    )

num_pairs = min(len(test_examples), len(pred_examples), MAX_EXAMPLES)
results = []

for i in tqdm(range(num_pairs), desc="Evaluating predictions", dynamic_ncols=True):
    test_ex = test_examples[i]
    pred_ex = pred_examples[i]

    prediction = pred_ex.get("prediction", "")
    tests = test_ex.get("translated_test_cases", [])

    eval_result = run_tests(prediction, tests)

    results.append({
        "idx": i,
        "compiled": eval_result["compiled"],
        "tests_passed": eval_result["passed"],
        "tests_total": eval_result["total"],
        "had_no_tests": len(tests) == 0,
        "stderr": eval_result["stderr"][:2000],
    })

    if (i + 1) % PRINT_EVERY == 0 or (i + 1) == num_pairs:
        compiled_so_far = sum(r["compiled"] for r in results)
        passed_so_far = sum(r["tests_passed"] for r in results)
        total_so_far = sum(r["tests_total"] for r in results)

        compile_rate_so_far = compiled_so_far / len(results) if results else 0.0
        test_rate_so_far = passed_so_far / total_so_far if total_so_far else 0.0

        print(
            f"\n[{i + 1}/{num_pairs}] "
            f"compile rate so far: {compiled_so_far}/{len(results)} = {compile_rate_so_far:.3f} | "
            f"test pass rate so far: {passed_so_far}/{total_so_far} = {test_rate_so_far:.3f}"
        )

num_examples = len(results)
compile_successes = sum(r["compiled"] for r in results)
total_passed = sum(r["tests_passed"] for r in results)
total_tests = sum(r["tests_total"] for r in results)

print("===== OVERALL EVALUATION =====")
if num_examples:
    print(f"Examples evaluated: {num_examples}")
    print(f"Compile success: {compile_successes}/{num_examples} = {compile_successes/num_examples:.3f}")
else:
    print("Examples evaluated: 0")

if total_tests:
    print(f"Functional test pass rate: {total_passed}/{total_tests} = {total_passed/total_tests:.3f}")
else:
    print("Functional test pass rate: N/A")

print("\n===== PER-EXAMPLE SUMMARY =====")
for r in results:
    print(f"\n--- Example {r['idx']} ---")
    print("Compiled:", r["compiled"])
    print(f"Tests passed: {r['tests_passed']}/{r['tests_total']}")
    if not r["compiled"]:
        print("Error:")
        print(r["stderr"])

summary = {
    "examples_evaluated": num_examples,
    "compile_successes": compile_successes,
    "compile_rate": (compile_successes / num_examples) if num_examples else None,
    "tests_passed": total_passed,
    "tests_total": total_tests,
    "functional_pass_rate": (total_passed / total_tests) if total_tests else None,
}

output = {
    "summary": summary,
    "per_example_results": results
}

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2)

print(f"\nSaved evaluation results to {RESULTS_PATH}")